In [0]:
class DataLoader:

    def __init__(self):
        self.catalog = "dev"
        self.db = "spark_db"
        self.volumne_name = "datasets"
        self.owner = "ashiesinha"
        self.repo = "PySpark"

    def cleanup_dir(self, dest_path):
        print(f"Cleaning {dest_path}...", end="")
        dbutils.fs.rm(dest_path, recurse=True)
        dbutils.fs.mkdirs(dest_path)
        print("Done")

    def clean_ingest_data(self, source, dest):
        from concurrent.futures import ThreadPoolExecutor

        import requests

        dest_path = (
            f"/Volumes/{self.catalog}/{self.db}/{self.volumne_name}/{dest}"
        )

        #api_url = f"https://api.github.com/repos/{self.owner}/{self.repo}/contents/{source}"
        api_url = f"https://api.github.com/repos/LearningJournal/scholarnest_datasets/contents/spark_programming/{source}"
        res = requests.get(api_url)
        response = res.json()

        # Check if GitHub returned an error object instead of a file list
        if not isinstance(response, list):
            raise RuntimeError(
                f"GitHub API Error ({res.status_code}): {response.get('message', response)}"
            )

        self.cleanup_dir(dest_path)

        # Filter and sort files
        files = [f for f in response if f["type"] == "file"]
        files = sorted(files, key=lambda x: x["name"])

        def download_file(file):
            file_url = file["download_url"]
            file_name = file["name"]
            file_path = f"{dest_path}/{file_name}"
            print(f"Downloading {file_name}...", end="")
            r = requests.get(file_url)
            with open(file_path, "wb") as f:
                f.write(r.content)
            print("Done")

        with ThreadPoolExecutor(max_workers=10) as executor:
            executor.map(download_file, files)